# 06 fairness and Explainability

In [3]:
import os, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_score, recall_score,
                             precision_recall_curve)
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore", message=".*ill-defined.*")
SEED = 42
DATA = "C:/Users/Siddhesh/Downloads/bank-customer-churn/data/raw/bank_churn.csv"
pd.set_option("display.width", 150)

if "pred" not in globals():
    print("Rebuilding the modeling setup (same code as notebook 05)...")
    raw = pd.read_csv(DATA)
    assert raw["CustomerId"].is_unique
    y = raw["Exited"].astype(int)
    df = raw.drop(columns=["RowNumber", "CustomerId", "Surname", "Exited"])
    df["HasZeroBalance"]     = (df["Balance"] == 0).astype(int)
    df["BalanceToSalary"]    = df["Balance"] / df["EstimatedSalary"].replace(0, np.nan)
    df["AgeBand"]            = pd.cut(df["Age"], [0, 29, 39, 49, 59, 120],
                                      labels=["<30", "30-39", "40-49", "50-59", "60+"]).astype(str)
    df["ProductsGroup"]      = df["NumOfProducts"].clip(upper=3).astype(str).replace({"3": "3+"})
    df["TenureYoungAccount"] = (df["Tenure"] <= 1).astype(int)
    X_tr, X_te, y_tr, y_te = train_test_split(df, y, test_size=0.2, stratify=y, random_state=SEED)

    def build_models(num, cat):
        def prep(scale):
            steps = [("imp", SimpleImputer(strategy="median"))] + ([("sc", StandardScaler())] if scale else [])
            return ColumnTransformer([("num", Pipeline(steps), num),
                                      ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])
        return {
            "Baseline (majority)": Pipeline([("p", prep(False)), ("m", DummyClassifier(strategy="prior"))]),
            "LogReg": Pipeline([("p", prep(True)), ("m", LogisticRegression(max_iter=2000, class_weight="balanced"))]),
            "RandomForest": Pipeline([("p", prep(False)), ("m", RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                      class_weight="balanced", n_jobs=-1, random_state=SEED))]),
            "HistGB": Pipeline([("p", prep(False)), ("m", HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05,
                                max_iter=300, class_weight="balanced", random_state=SEED))]),
        }
    num = X_tr.select_dtypes("number").columns.tolist()
    cat = X_tr.select_dtypes(exclude="number").columns.tolist()
    models = build_models(num, cat)
    cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
    scores = {n: cross_validate(m, X_tr, y_tr, cv=cv, scoring="average_precision")["test_score"].mean()
              for n, m in models.items() if not n.startswith("Baseline")}
    best_name = max(scores, key=scores.get); best = models[best_name]
    oof = cross_val_predict(best, X_tr, y_tr, cv=cv, method="predict_proba")[:, 1]
    prec, rec, thr = precision_recall_curve(y_tr, oof)
    threshold = float(thr[(2*prec[:-1]*rec[:-1]/(prec[:-1]+rec[:-1]+1e-9)).argmax()])
    best.fit(X_tr, y_tr); p = best.predict_proba(X_te)[:, 1]; pred = (p >= threshold).astype(int)

print("Best model:", best_name, "| threshold:", round(threshold, 3))
print("Test customers:", len(X_te), "| overall churn rate:", round(y_te.mean(), 3), "| overall flagged:", round(pred.mean(), 3))

Best model: HistGB | threshold: 0.615
Test customers: 2000 | overall churn rate: 0.204 | overall flagged: 0.22
